# ДЗ 3

Марковский Артём Владиславович, 466636, J3310

Вариант: Программист, разработчик на Go

In [1]:
import gc
import json
import math
import pathlib
import subprocess
import sys
import threading
import time
from collections import Counter, defaultdict
from importlib.metadata import version
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
from boto3.s3.transfer import TransferConfig
from IPython import get_ipython
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

from common import ACCESS_KEY, BUCKET, CSV_KEY, SECRET_KEY, s3_client

PROJECT = pathlib.Path.cwd()
DATA = PROJECT.parent / "big" / "vacancies_2024.csv"
DISKS = PROJECT.parent / "rustfs"
OUT = PROJECT / "out"
OUT.mkdir(exist_ok=True)

N_DISKS = 12
PART_SIZE = 16 * 1024 * 1024
MiB = 1024 * 1024
BLUE = "blue"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("python", sys.version.split()[0], "duckdb", version("duckdb"), "polars", version("polars"), "pandas", version("pandas"))
print("ядер", psutil.cpu_count(), "памяти, ГиБ", round(psutil.virtual_memory().total / 1024 ** 3, 1))

python 3.13.7 duckdb 2.0.0 polars 2.1.0 pandas 3.1.0
ядер 16 памяти, ГиБ 15.2


Замер ячейки: %%measure N название выполняет ячейку N раз и на каждом запуске считает время и память только этой ячейки. Перед каждым запуском перезапускаю RustFS и чищу мусор Python, чтобы все попытки стартовали одинаково, это время в замер не входит

In [2]:
measurements: list[dict[str, str | float]] = []


def used_memory() -> int:
    kernel = psutil.Process()
    total = kernel.memory_info().rss
    
    for child in kernel.children(recursive=True):
        try:
            total += child.memory_info().rss
        except psutil.Error:
            pass
            
    return total


def fresh_start() -> None:
    run_command("docker", "restart", "rustfs")
    
    for attempt in range(90):
        try:
            s3.get_object(Bucket=BUCKET, Key=CSV_KEY, Range="bytes=0-1048575")["Body"].read()
            break
        except Exception:
            time.sleep(2)
            
    time.sleep(5)
    gc.collect()


@register_cell_magic
def measure(line: str, cell: str) -> None:
    words = line.split()
    repeats = int(words[0])
    label = " ".join(words[1:])
    
    for attempt in range(1, repeats + 1):
        fresh_start()
        before = used_memory()
        peak = [before]
        running = [True]

        def watch() -> None:
            while running[0]:
                peak[0] = max(peak[0], used_memory())
                time.sleep(0.1)

        watcher = threading.Thread(target=watch)
        watcher.start()
        started = time.perf_counter()
        result = get_ipython().run_cell(cell)
        seconds = time.perf_counter() - started
        running[0] = False
        watcher.join()
        row: dict[str, str | float] = {"способ": label, "попытка": attempt, "время, с": round(seconds, 1),
                   "пик памяти, МиБ": round(peak[0] / MiB), "прирост памяти, МиБ": round((peak[0] - before) / MiB)}
        measurements.append(row)
        print(label, "попытка", attempt, "время", row["время, с"], "с, пик памяти", row["пик памяти, МиБ"],
                  "МиБ, прирост", row["прирост памяти, МиБ"], "МиБ")
                  
        if not result.success:
            raise RuntimeError("ячейка упала")

## 1. Хранилище

Поднимаю RustFS

In [3]:
def run_command(*arguments: str) -> str:
    result = subprocess.run(arguments, capture_output=True, text=True, encoding="utf-8", errors="replace")
    
    return (result.stdout + result.stderr).strip()

    
for disk in range(1, N_DISKS + 1):
    (DISKS / ("disk" + str(disk))).mkdir(parents=True, exist_ok=True)
    
status = run_command("docker", "ps", "-a", "--filter", "name=^rustfs$", "--format", "{{.Status}}")

if status == "":
    command = ["docker", "run", "-d", "--name", "rustfs", "-p", "9000:9000", "-p", "9001:9001",
               "-e", "RUSTFS_ACCESS_KEY=" + ACCESS_KEY, "-e", "RUSTFS_SECRET_KEY=" + SECRET_KEY,
               "-e", "RUSTFS_VOLUMES=/data/disk{1..." + str(N_DISKS) + "}",
               "-e", "RUSTFS_ADDRESS=:9000", "-e", "RUSTFS_CONSOLE_ADDRESS=:9001",
               "-e", "RUSTFS_CONSOLE_ENABLE=true", "-e", "RUSTFS_UNSAFE_BYPASS_DISK_CHECK=true"]
               
    for disk in range(1, N_DISKS + 1):
        name = "disk" + str(disk)
        command += ["--mount", "type=bind,source=" + str(DISKS / name) + ",target=/data/" + name]
        
    command.append("rustfs/rustfs:latest")
    print(run_command(*command))
elif not status.startswith("Up"):
    print(run_command("docker", "start", "rustfs"))

print(run_command("docker", "ps", "--filter", "name=^rustfs$"))

print(run_command("docker", "exec", "rustfs", "rustfs", "--version").splitlines()[0])
print(run_command("docker", "inspect", "rustfs", "--format", "{{range .Config.Env}}{{println .}}{{end}}"))

CONTAINER ID   IMAGE                  COMMAND                  CREATED        STATUS         PORTS                                                             NAMES
5fb9686150ff   rustfs/rustfs:latest   "/entrypoint.sh rust…"   22 hours ago   Up 7 minutes   0.0.0.0:9000-9001->9000-9001/tcp, [::]:9000-9001->9000-9001/tcp   rustfs
rustfs 1.0.1
RUSTFS_VOLUMES=/data/disk{1...12}
RUSTFS_ADDRESS=:9000
RUSTFS_CONSOLE_ADDRESS=:9001
RUSTFS_CONSOLE_ENABLE=true
RUSTFS_UNSAFE_BYPASS_DISK_CHECK=true
RUSTFS_ACCESS_KEY=rustfsadmin
RUSTFS_SECRET_KEY=rustfsadmin
PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
RUSTFS_CONSOLE_CORS_ALLOWED_ORIGINS=*
RUSTFS_OBS_LOGGER_LEVEL=warn
RUSTFS_OBS_LOG_DIRECTORY=/logs
RUSTFS_OBS_ENVIRONMENT=production


In [4]:
s3 = s3_client()

for attempt in range(60):
    try:
        s3.list_buckets()
        break
    except Exception:
        time.sleep(2)
        
if BUCKET not in [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]]:
    s3.create_bucket(Bucket=BUCKET)
print("бакеты:", [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]])

бакеты: ['hh-lake']


Загружаю через S3 API, multipart upload частями по 16 МиБ в 8 потоков, время пишу в out/upload_log.json, при повторном запуске ячейка берёт его оттуда

In [5]:
size = DATA.stat().st_size
upload_log = OUT / "upload_log.json"
listed = s3.list_objects_v2(Bucket=BUCKET, Prefix=CSV_KEY)
uploaded = listed["Contents"][0]["Size"] if listed["KeyCount"] else 0

if uploaded == size and upload_log.exists():
    print("файл уже в бакете, лог загрузки:")
    print(upload_log.read_text(encoding="utf-8"))
else:
    lock = threading.Lock()
    bar = tqdm(total=size, unit="B", desc="upload")

    def progress(sent: int) -> None:
        with lock:
            bar.update(sent)

    started = time.perf_counter()
    transfer = TransferConfig(multipart_threshold=PART_SIZE, multipart_chunksize=PART_SIZE, max_concurrency=8)
    s3.upload_file(str(DATA), BUCKET, CSV_KEY, Callback=progress, Config=transfer)
    bar.close()
    seconds = time.perf_counter() - started
    log = {"key": "s3://" + BUCKET + "/" + CSV_KEY, "bytes": size, "seconds": round(seconds, 1),
           "MiB_per_s": round(size / MiB / seconds, 1), "parts": math.ceil(size / PART_SIZE)}
    upload_log.write_text(json.dumps(log, indent=1), encoding="utf-8")
    print(log)

файл уже в бакете, лог загрузки:
{
  "key": "s3://hh-lake/raw/vacancies_2024.csv",
  "bytes": 24970431974,
  "seconds": 744.0,
  "MiB_per_s": 32.0,
  "part_size": 16777216,
  "parts": 1489,
  "finished": "2026-10-26 16:23:36"
}


Проверка: лежит ли объект в бакете hh-lake и сколько места заняли папки-диски

In [6]:
head = s3.head_object(Bucket=BUCKET, Key=CSV_KEY)
print("s3://" + BUCKET + "/" + CSV_KEY, head["ContentLength"], "байт, локальный файл", size, "байт, совпадает:", head["ContentLength"] == size)

for item in s3.list_objects_v2(Bucket=BUCKET)["Contents"]:
    print(item["Key"], item["Size"])
    
disk_rows = []

for disk in range(1, N_DISKS + 1):
    files = [path for path in (DISKS / ("disk" + str(disk))).rglob("*") if path.is_file()]
    
    disk_rows.append({"диск": "disk" + str(disk), "файлов": len(files), "байт": sum(path.stat().st_size for path in files)})
    
disks = pd.DataFrame(disk_rows)
print(disks)
print("всего на дисках", disks["байт"].sum(), "байт, это", round(disks["байт"].sum() / size, 3), "от размера CSV")

s3://hh-lake/raw/vacancies_2024.csv 24970431974 байт, локальный файл 24970431974 байт, совпадает: True
parquet/skills.parquet 4852514
parquet/vacancies/data_0.parquet 233745515
raw/vacancies_2024.csv 24970431974
results/duckdb/skills_k1.csv 16467
results/duckdb/skills_k2.csv 307411
results/duckdb/skills_k3.csv 2510269
results/mapreduce/skills_k1.csv 16463
results/mapreduce/skills_k2.csv 307147
results/mapreduce/skills_k3.csv 2507949
results/polars/skills_k1.csv 16463
results/polars/skills_k2.csv 307147
results/polars/skills_k3.csv 2507949
variant/go_vacancies.parquet 142146
      диск  файлов        байт
0    disk1    3050  3153300016
1    disk2    3049  3153300016
2    disk3    3049  3153300016
3    disk4    3049  3153300016
4    disk5    3049  3153300016
5    disk6    3049  3153300016
6    disk7    3049  3153300016
7    disk8    3049  3153300016
8    disk9    3049  3153300016
9   disk10    3049  3153300016
10  disk11    3049  3153300016
11  disk12    3049  3153300016
всего на дисках 